<a href="https://colab.research.google.com/github/msankar/cheat-at-search/blob/main/2c_Query_Categories_Search_First.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Query to get candidates BEFORE classification

<small>
(from <a href="http://maven.com/softwaredoug/cheat-at-search">Cheat at Search with Agents</a> training course by Doug Turnbull.)
</small>

It might not be feasible to do a lot of LLM classification unless we can use a smaller model with fewer tokens. One approach is to search and THEN classify into some of the matching categories. That's what we do here.

This might let us get some of the benefit with less of the downside.


## Boilerplate

Install deps, mount GDrive, prompt for your OpenAI Key (placed in your GDrive), and import needed cheat at search helpers.

We cover this extensively in the [synonyms notebook](https://colab.research.google.com/drive/1aUCvcBa1YdmsbIgYc74jlknl9_iRotp1) walkthrough

In [ ]:
!pip install git+https://github.com/softwaredoug/cheat-at-search.git
from cheat_at_search.data_dir import mount
mount(use_gdrive=True)
from cheat_at_search.search import run_strategy, ndcgs, ndcg_delta, vs_ideal
from cheat_at_search.wands_data import products, judgments, labeled_query_products

products

  Cloning https://github.com/softwaredoug/cheat-at-search.git to /tmp/pip-req-build-r0lot8ab
  Running command git clone --filter=blob:none --quiet https://github.com/softwaredoug/cheat-at-search.git /tmp/pip-req-build-r0lot8ab
  Resolved https://github.com/softwaredoug/cheat-at-search.git to commit edca45d487123153a7db11d7cd2c7bc088842e07
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 745.3/745.3 kB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 10.6 MB/s eta 0:00:00
  Created wheel for cheat_at_search: filename=cheat_at_search-0.1.0-py3-none-any.whl size=1453000 sha256=b3b100a3b1d65b933ef2

,product_id,product_name,product_class,category hierarchy,product_description,product_features,rating_count,average_rating,review_count,features,doc_id,title,description,category,sub_category,cat_subcat
0,0,solid wood platform bed,Beds,Furniture / Bedroom Furniture / Beds & Headboa...,"good , deep sleep can be quite difficult to ha...",overallwidth-sidetoside:64.7|dsprimaryproducts...,15.0,4.5,15.0,"[overallwidth-sidetoside:64.7, dsprimaryproduc...",0,solid wood platform bed,"good , deep sleep can be quite difficult to ha...",Furniture,Bedroom Furniture,Furniture / Bedroom Furniture
1,1,all-clad 7 qt . slow cooker,Slow Cookers,Kitchen & Tabletop / Small Kitchen Appliances ...,"create delicious slow-cooked meals , from tend...",capacityquarts:7|producttype : slow cooker|pro...,100.0,2.0,98.0,"[capacityquarts:7, producttype : slow cooker, ...",1,all-clad 7 qt . slow cooker,"create delicious slow-cooked meals , from tend...",Kitchen & Tabletop,Small Kitchen Appliances,Kitchen & Tabletop / Small Kitchen Appliances
2,2,all-clad electrics 6.5 qt . slow cooker,Slow Cookers,Kitchen & Tabletop / Small Kitchen Appliances ...,prepare home-cooked meals on any schedule with...,features : keep warm setting|capacityquarts:6....,208.0,3.0,181.0,"[features : keep warm setting, capacityquarts:...",2,all-clad electrics 6.5 qt . slow cooker,prepare home-cooked meals on any schedule with...,Kitchen & Tabletop,Small Kitchen Appliances,Kitchen & Tabletop / Small Kitchen Appliances
3,3,all-clad all professional tools pizza cutter,"Slicers, Peelers And Graters",Browse By Brand / All-Clad,this original stainless tool was designed to c...,overallwidth-sidetoside:3.5|warrantylength : l...,69.0,4.5,42.0,"[overallwidth-sidetoside:3.5, warrantylength :...",3,all-clad all professional tools pizza cutter,this original stainless tool was designed to c...,Browse By Brand,All-Clad,Browse By Brand / All-Clad
4,4,baldwin prestige alcott passage knob with roun...,Door Knobs,Home Improvement / Doors & Door Hardware / Doo...,the hardware has a rich heritage of delivering...,compatibledoorthickness:1.375 '' |countryofori...,70.0,5.0,42.0,"[compatibledoorthickness:1.375 '' , countryofo...",4,baldwin prestige alcott passage knob with roun...,the hardware has a rich heritage of delivering...,Home Improvement,Doors & Door Hardware,Home Improvement / Doors & Door Hardware
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42989,42989,malibu pressure balanced diverter fixed shower...,Shower Panels,Home Improvement / Bathroom Remodel & Bathroom...,the malibu pressure balanced diverter fixed sh...,producttype : shower panel|spraypattern : rain...,3.0,4.5,2.0,"[producttype : shower panel, spraypattern : ra...",42989,malibu pressure balanced diverter fixed shower...,the malibu pressure balanced diverter fixed sh...,Home Improvement,Bathroom Remodel & Bathroom Fixtures,Home Improvement / Bathroom Remodel & Bathroom...
42990,42990,emmeline 5 piece breakfast dining set,Dining Table Sets,Furniture / Kitchen & Dining Furniture / Dinin...,,basematerialdetails : steel| : gray wood|ofhar...,1314.0,4.5,864.0,"[basematerialdetails : steel, : gray wood, of...",42990,emmeline 5 piece breakfast dining set,,Furniture,Kitchen & Dining Furniture,Furniture / Kitchen & Dining Furniture
42991,42991,maloney 3 piece pub table set,Dining Table Sets,Furniture / Kitchen & Dining Furniture / Dinin...,this pub table set includes 1 counter height t...,additionaltoolsrequirednotincluded : power dri...,49.0,4.0,41.0,[additionaltoolsrequirednotincluded : power dr...,42991,maloney 3 piece pub table set,this pub table set includes 1 counter height t...,Furniture,Kitchen & Dining Furniture,Furniture / Kitchen & Dining Furniture
42992,42992,fletcher 27.5 '' wide polyester armchair,Teen Lounge Furniture|Accent Chairs,Furniture / Living Room Furniture / Chairs & S...,"bring iconic , modern style to your space in a...",legmaterialdetails : rubberwood|backheight-sea...,1746.0,4.5,1226.0,"[legmateri

## Search function for search -> categorize

Note in this notebook we do two searches. One at classification time, a second the actual query. This recreates the search we might perform offline, during some batch process of query classification.

In [ ]:
from searcharray import SearchArray
from cheat_at_search.tokenizers import snowball_tokenizer
import numpy as np
import pandas as pd

product_name_index = SearchArray.index(products['product_name'], snowball_tokenizer)
product_desc_index  = SearchArray.index(products['product_description'], snowball_tokenizer)

def matching_classifications(query, k=10):
    """Issue a search then get all category hierarchy for matches."""
    tokenized = snowball_tokenizer(query)
    bm25_scores = np.zeros(len(products))
    for token in tokenized:
        bm25_scores += product_name_index.score(token) * 7
        bm25_scores += product_desc_index.score(token) * 4
    # Get top 300 bm25_scores, set all others to 0
    # bm25_scores[bm25_scores.argsort()[:-300]] = 0

    matches = products[bm25_scores > 0]
    top_10 = matches.groupby('category hierarchy')['product_id'].nunique().sort_values(ascending=False).head(25)
    return top_10


2026-05-20 12:54:28,005 - searcharray.indexing - INFO - Indexing begins w/ 4 workers


INFO:searcharray.indexing:Indexing begins w/ 4 workers


2026-05-20 12:54:28,013 - searcharray.indexing - INFO - 0 Batch Start tokenization


INFO:searcharray.indexing:0 Batch Start tokenization


2026-05-20 12:54:28,016 - searcharray.indexing - INFO - Tokenizing 42994 documents


...

INFO:searcharray.indexing:Batch tokenization complete


2026-05-20 12:54:43,108 - searcharray.indexing - INFO - (main thread) Processing 1 batch results


INFO:searcharray.indexing:(main thread) Processing 1 batch results


2026-05-20 12:54:43,249 - searcharray.indexing - INFO - Indexing from tokenization complete


INFO:searcharray.indexing:Indexing from tokenization complete


In [ ]:
matching_classifications("couch")

,product_id
category hierarchy,
Décor & Pillows / Decorative Pillows & Blankets / Throw Pillows,49
Furniture / Living Room Furniture / Sectionals,28
Furniture / Living Room Furniture / Coffee Tables & End Tables / End & Side Tables,26
Furniture / Living Room Furniture / Sofas,25
Furniture / Living Room Furniture / Coffee Tables & End Tables / Coffee Tables,24
Décor & Pillows / Decorative Pillows & Blankets / Throw Pillows / Blue Throw Pillows,15
Furniture / Living Room Furniture / Coffee Tables & End Tables / Coffee Tables / Rectangle Coffee Tables,14
Furniture / Living Room Furniture / Chairs & Seating / Chaise Lounge Chairs,9
Furniture / Bedroom Furniture / Daybeds,9


## Dynamically build classification model

Here we dynamically bulid the classification model from the labels. Notice below we just test with 'a', 'b', 'c'. But in reality we'de take the list of top N classifications from above to classify with.

In [ ]:

from enum import Enum
from pydantic import BaseModel, Field, ConfigDict, create_model

def make_classifier_model(labels: list[str]) -> type[BaseModel]:
    # Remove any label with illegal chars
    illegal_chars = '"'
    labels = [label for label in labels if all(c not in illegal_chars for c in label)]

    # Use safe member names; preserve the *actual* label as the Enum value.
    members = {f"v{i}": label for i, label in enumerate(labels)}
    LabelEnum = Enum("LabelEnum", members)

    # Make a model whose field is restricted to those enum values
    Model = create_model(
        "QueryClassification",
        __doc__="A classification of the query.",
        classification=(list[LabelEnum], Field("Which classifications might make sense for the query. Leave blank if none.")),
        __config__=ConfigDict(use_enum_values=True),

    )
    # print(Model.model_json_schema())
    return Model


model = make_classifier_model(["a", "b", "c"])
model.model_json_schema()

{'$defs': {'LabelEnum': {'enum': ['a', 'b', 'c'],
   'title': 'LabelEnum',
   'type': 'string'}},
 'description': 'A classification of the query.',
 'properties': {'classification': {'default': 'Which classifications might make sense for the query. Leave blank if none.',
   'items': {'$ref': '#/$defs/LabelEnum'},
   'title': 'Classification',
   'type': 'array'}},
 'title': 'QueryClassification',
 'type': 'object'}

In [ ]:
# Remove cache as needed
!rm /content/drive/MyDrive/cheat-at-search-data/search_then_classify/cache.json

In [ ]:
from re import sub
from cheat_at_search.data_dir import key_for_provider, ensure_data_subdir
from openai import OpenAI
import json

OPENAI_KEY = key_for_provider("openai")

openai = OpenAI(api_key=OPENAI_KEY)

cache_dir = ensure_data_subdir("search_then_classify")
try:
    with open(cache_dir / "cache.json", 'rb') as f:
        cache = json.load(f)
except FileNotFoundError:
    cache = {}


class StructuredClassification(BaseModel):
    classifications: list[str]

    categories: list[str]

    sub_categories: list[str]

    cat_subcat: list[str]


def get_category_subcategory(classifications) -> StructuredClassification:
    categories = set()
    sub_categories = set()
    cat_subcat = set()
    if not classifications:
        return StructuredClassification(classifications=classifications,
                                        categories=[],
                                        sub_categories=[],
                                        cat_subcat=[])
    for classification in classifications:
        splits = classification.split('/')
        if splits:
            category = splits[0].strip()
            sub_category = splits[1].strip() if len(splits) > 1 else "No SubCategory Found"
            categories.add(category)
            sub_categories.add(sub_category)
            cat_subcat.add(f"{category} / {sub_category}")
        return StructuredClassification(classifications=classifications,
                                        categories=list(categories),
                                        sub_categories=list(sub_categories),
                                        cat_subcat=list(cat_subcat))

def enrich_query(query):
    """Use responses.parse with the dynamic classifaciots schema to classify query."""
    if query in cache:
        return get_category_subcategory(cache[query])
    best_classifications = matching_classifications(query)
    if len(best_classifications) == 0:
        return get_category_subcategory([])
    ClassificationModel = make_classifier_model(best_classifications.index.tolist())
    system_prompt = "Map user furniture search queries to classifications"
    prompt = f"""
        Return all relevant classifications for this search query
        If none / not sure return an empty list

        Query: {query}
    """
    resp = openai.responses.parse(
        model="gpt-4.1-nano",
        input=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt}
        ],
        text_format=ClassificationModel
    )
    usage = resp.usage
    print(f"Input tokens {usage.input_tokens}, output tokens {usage.output_tokens}")
    classification = resp.output_parsed
    cache[query] = classification.classification
    with open(cache_dir / "cache.json", "w") as f:
        json.dump(cache, f)
    return get_category_subcategory(classification.classification)


enrich_query("sofa")

Input tokens 485, output tokens 14


StructuredClassification(classifications=['Furniture / Living Room Furniture / Sofas'], categories=['Furniture'], sub_categories=['Living Room Furniture'], cat_subcat=['Furniture / Living Room Furniture'])

In [ ]:
labeled_query_products[['query', 'doc_id', 'category', 'grade']]

,query,doc_id,category,grade
0,salon chair,17,Home Improvement,0.0
1,salon chair,63,Furniture,1.0
2,salon chair,65,Furniture,1.0
3,salon chair,95,Furniture,1.0
4,salon chair,130,Furniture,1.0
...,...,...,...,...
231868,rack glass,42869,Kitchen & Tabletop,0.0
231869,rack glass,42870,Kitchen & Tabletop,0.0
231870,rack glass,42871,Kitchen & Tabletop,0.0
231871,rack glass,42872,,0.0


## Run Category search strategy with classifier

Our search strategy here is identical, **however note** when the LLM does not predict a category / sub category we do not provide a boost.

In [ ]:
from searcharray import SearchArray
from cheat_at_search.tokenizers import snowball_tokenizer
from cheat_at_search.strategy.strategy import SearchStrategy
import numpy as np


class CategorySearch(SearchStrategy):
    def __init__(self, products, query_to_cat,
                 name_boost=9.3,
                 description_boost=4.1,
                 category_boost=10,
                 sub_category_boost=5,
                 cat_subcat_boost=10):
        super().__init__(products)
        self.index = products
        self.index['product_name_snowball'] = SearchArray.index(
            products['product_name'], snowball_tokenizer)
        self.index['product_description_snowball'] = SearchArray.index(
            products['product_description'], snowball_tokenizer)

        cat_split = products['category hierarchy'].fillna('').str.split("/")

        products['category'] = cat_split.apply(
            lambda x: x[0].strip() if len(x) > 0 else ""
        )
        products['subcategory'] = cat_split.apply(
            lambda x: x[1].strip() if len(x) > 1 else ""
        )
        self.index['category_snowball'] = SearchArray.index(
            products['category'], snowball_tokenizer
        )
        self.index['subcategory_snowball'] = SearchArray.index(
            products['subcategory'], snowball_tokenizer
        )

        self.index['cat_subcat'] = products['category'] + products['subcategory']
        self.index['cat_subcat'] = self.index['cat_subcat'].fillna('')
        self.index['cat_subcat_snowball'] = SearchArray.index(
            self.index['cat_subcat'], snowball_tokenizer
        )

        self.query_to_cat = query_to_cat
        self.name_boost = name_boost
        self.description_boost = description_boost
        self.category_boost = category_boost
        self.sub_category_boost = sub_category_boost
        self.cat_subcat_boost = cat_subcat_boost

    def search(self, query, k=10):
        """Dumb baseline lexical search, but add a constant boost when
           the desired category or subcategory"""
        bm25_scores = np.zeros(len(self.index))
        structured = self.query_to_cat(query)
        tokenized = snowball_tokenizer(query)

        print(structured)

        # ****
        # Baseline BM25 search from before
        for token in tokenized:
            bm25_scores += self.index['product_name_snowball'].array.score(token) * self.name_boost
            bm25_scores += self.index['product_description_snowball'].array.score(
                token) * self.description_boost

        # ****
        # If there's a subcategory, boost that by a constant amount
        for sub_category in structured.sub_categories:
            tokenized_subcategory = snowball_tokenizer(sub_category)
            subcategory_match = np.zeros(len(self.index))
            if tokenized_subcategory:
                subcategory_match = self.index['subcategory_snowball'].array.score(tokenized_subcategory) > 0
            bm25_scores[subcategory_match] += self.sub_category_boost

        # ****
        # If there's a category, boost that by a constant amount
        for category in structured.categories:
            print(category)
            tokenized_category = snowball_tokenizer(category)
            category_match = np.zeros(len(self.index))
            if tokenized_category:
                category_match = self.index['category_snowball'].array.score(tokenized_category) > 0
            bm25_scores[category_match] += self.category_boost

        # ***
        # If they occur togethere, boost by a constant amount
        for cat_subcat in structured.cat_subcat:
            tokenized_cat_subcat = snowball_tokenizer(cat_subcat)
            cat_subcat_match = np.zeros(len(self.index))
            if tokenized_cat_subcat:
                cat_subcat_match = self.index['cat_subcat_snowball'].array.score(tokenized_cat_subcat) > 0
            bm25_scores[cat_subcat_match] += self.cat_subcat_boost

        top_k = np.argsort(-bm25_scores)[:k]
        scores = bm25_scores[top_k]

        return top_k, scores

categorized_search = CategorySearch(products, enrich_query)
categorized_search.search('medium clips')

2026-05-20 12:54:54,001 - searcharray.indexing - INFO - Indexing begins w/ 4 workers


INFO:searcharray.indexing:Indexing begins w/ 4 workers


2026-05-20 12:54:54,007 - searcharray.indexing - INFO - 0 Batch Start tokenization


INFO:searcharray.indexing:0 Batch Start tokenization


2026-05-20 12:54:54,009 - searcharray.indexing - INFO - Tokenizing 42994 documents


...

INFO:searcharray.indexing:(main thread) Processing 1 batch results


2026-05-20 12:55:04,680 - searcharray.indexing - INFO - Indexing from tokenization complete


INFO:searcharray.indexing:Indexing from tokenization complete


Input tokens 441, output tokens 11
classifications=['Clips/Clamps'] categories=['Clips'] sub_categories=['Clamps'] cat_subcat=['Clips / Clamps']
Clips


(array([36813, 37397, 34050, 36809, 37396, 37398, 36810, 11295, 37007,
        16370]),
 array([85.35794735, 84.99754143, 68.97158527, 68.62535572, 67.39479828,
        66.0164814 , 64.83574677, 62.97011852, 62.32560349, 60.5767231 ]))

In [ ]:
from cheat_at_search.wands_data import judgments, graded_bm25

In [ ]:
graded_categorized = run_strategy(categorized_search, judgments)
graded_categorized

Searching:   0%|          | 1/480 [00:01<11:50,  1.48s/it]

Input tokens 526, output tokens 5
classifications=[] categories=[] sub_categories=[] cat_subcat=[]


Searching:   0%|          | 2/480 [00:03<16:09,  2.03s/it]

Input tokens 591, output tokens 28
classifications=['Home Improvement / Flooring, Walls & Ceiling / Walls & Ceilings / Peel & Stick Backsplash Tile'] categories=['Home Improvement'] sub_categories=['Flooring, Walls & Ceiling'] cat_subcat=['Home Improvement / Flooring, Walls & Ceiling']
Home Improvement


Searching:   1%|          | 3/480 [00:07<21:00,  2.64s/it]

...

Input tokens 471, output tokens 17
classifications=['Furniture / Bedroom Furniture / Beds & Headboards / Beds'] categories=['Furniture'] sub_categories=['Bedroom Furniture'] cat_subcat=['Furniture / Bedroom Furniture']
Furniture


Searching: 100%|██████████| 480/480 [10:47<00:00,  1.02it/s]

Input tokens 576, output tokens 5
classifications=[] categories=[] sub_categories=[] cat_subcat=[]


Searching: 100%|██████████| 480/480 [10:48<00:00,  1.35s/it]


,product_id,product_name,product_class,category hierarchy,product_description,product_features,rating_count,average_rating,review_count,features,...,score,query,query_id,rank,grade,discounted_gain,idcg,dcg,ndcg,mrr
0,26644,international harvester melamine serving tray,Serving Dishes & Platters|Licensed Products,Kitchen & Tabletop / Tableware & Drinkware / S...,this case ih serving tray is a perfect additio...,overalllength-endtoend:11.5|pattern : no patte...,NaN,NaN,NaN,"[overalllength-endtoend:11.5, pattern : no pat...",...,69.232885,certified international melamine,73,1,1.0,1.000000,8.786905,2.828968,0.321953,0.0
1,9709,certified international gather oval baking dish,NaN,Kitchen & Tabletop / Cookware & Bakeware / Bak...,simple crockery with the look of hand-thrown p...,overalllength-endtoend:14.25|productcare : dis...,2.0,5.0,1.0,"[overalllength-endtoend:14.25, productcare : d...",...,62.629265,certified international melamine,73,2,1.0,0.500000,8.786905,2.828968,0.321953,0.0
2,26646,international harvester melamine cereal bowl,Dining Bowls|Licensed Products,Kitchen & Tabletop / Tableware & Drinkware / D...,this four pack of cereal or soup bowls feature...,color : red/white|productcare : dishwasher saf...,NaN,NaN,NaN,"[color : red/white, productcare : dishwasher s...",...,57.511242,certified international melamine,73,3,1.0,0.333333,8.786905,2.828968,0.321953,0.0
3,9707,certified international piazzette 3d villa sal...,Salt And Pepper Shakers / Grinders (Mills),Kitchen & Tabletop / Tableware & Drinkware / S...,,productweight:1.5|trend : novelty|productcare ...,15.0,5.0,12.0,"[productweight:1.5, trend : novelty, productca...",...,52.439243,certified international melamine,73,4,1.0,0.250000,8.786905,2.828968,0.321953,0.0
4,26645,international harvester melamine chip and dip ...,Serving Dishes & Platters|Licensed Products,Kitchen & Tabletop / Tableware & Drinkware / S...,"perfect for the case ih enthusiast , serve you...",producttype : chips and dip platter|overallwid...,NaN,NaN,NaN,"[producttype : chips and dip platter, overallw...",...,50.514072,certified international melamine,73,5,1.0,0.200000,8.786905,2.828968,0.321953,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4795,2207,flamingo magnet wall décor,Wall Décor,Sale / Closeout / Wall Accents / Blue Wall Acc...,this beautifully handcrafted flamingo wooden d...,productcare : wipe clean with a dry cloth . do...,NaN,NaN,NaN,[productcare : wipe clean with a dry cloth . d...,...,50.694689,flamingo,102,6,2.0,0.500000,8.786905,8.786905,1.000000,1.0
4796,17391,rondon a pink flamingo paper print,Kids Wall Décor,Baby & Kids / Baby & Kids Décor & Lighting / A...,perfect for the nursery or a little girl 's be...,gender : girl / woman+|primarymaterialdetails ...,1.0,5.0,1.0,"[gender : girl / woman+, primarymaterialdetail...",...,48.986856,flamingo,102,7,2.0,0.428571,8.786905,8.786905,1.000000,1.0
4797,3267,flamingos green by cat coquillette - print,Wall Art,Décor & Pillows / Art / All Wall Art / Green W...,showcasing a pair of flamingos encircled by ve...,holidayoccasion : no holiday|shape : square|co...,1.0,4.0,1.0,"[holidayoccasion : no holiday, shape : square,...",...,48.481298,flamingo,102,8,2.0,0.375000,8.786905,8.786905,1.000000,1.0
4798,13948,cervantes whimsical flamingo & pineapple micro...,Sheets And Sheet Sets,Bed & Bath / Bedding / Sheets & Pillowcases,a great whimsical print of flamingo and pineap...,fittedsheetlength-headtotoe:80|flatsheetwidth-...,12.0,5.0,11.0,"[fittedsheetlength-headtotoe:80, flatsheetwidt...",...,45.647358,flamingo,102,9,2.0,0.333333,8.786905,8.786905,1.000000,1.0


### Analyze results

We note:
1. NDCG down a bit from earlier
2. Though fewer queries have been harmed

In [ ]:
from cheat_at_search.search import graded_bm25

ndcgs(graded_bm25).mean(), ndcgs(graded_categorized).mean()

(np.float64(0.5411098691836396), np.float64(0.5506616086347829))

In [ ]:
deltas = ndcg_delta(graded_categorized, graded_bm25)
deltas

,ndcg
query,
drum picture,0.506435
bathroom vanity knobs,0.486519
non slip shower floor tile,0.477081
outdoor lounge chair,0.456442
modern outdoor furniture,0.365533
...,...
sugar canister,-0.224721
outdoor privacy wall,-0.345075
outdoor lounge cushions,-0.471391


In [ ]:
sig_improved = len(deltas[deltas > 0.1])
print(f"Num Significatly Improved: {sig_improved}")
deltas[deltas > 0.1]

Num Significatly Improved: 33


,ndcg
query,
drum picture,0.506435
bathroom vanity knobs,0.486519
non slip shower floor tile,0.477081
outdoor lounge chair,0.456442
modern outdoor furniture,0.365533
twin bed frame,0.359391
desk for kids,0.344262
outdoor light fixtures,0.320553
turquoise chair,0.310979


In [ ]:
sig_harmed = len(deltas[deltas < -0.1])
print(f"Num Significatly Harmed: {sig_harmed}")
print(f"Prop improved/harmed: {sig_improved / (sig_harmed + sig_improved)} | {sig_harmed / (sig_harmed + sig_improved)}")
deltas[deltas < -0.1]

Num Significatly Harmed: 14
Prop improved/harmed: 0.7021276595744681 | 0.2978723404255319


,ndcg
query,
zodiac pillow,-0.113806
papasan chair frame only,-0.123290
3/4 size mattress,-0.128031
bathroom freestanding cabinet,-0.155670
tall storage cabinet,-0.158063
adjustable height artist stool,-0.170709
wall design shelf,-0.187825
chair pillow cushion,-0.190625
sheffield home bath set,-0.219708


### Analyze a query

Let's look at a negative query to see how its harmed.

In [ ]:
QUERY = "chair pillow cushion"
graded_bm25[graded_bm25['query'] == QUERY][['product_name', 'product_description', 'category hierarchy', 'grade']]

,product_name,product_description,category hierarchy,grade
4660,replacement pillows outdoor lounge chair cushion,this replacement pillows outdoor lounge chair ...,NaN,2.0
4661,indoor/outdoor dining chair cushion and pillow...,go bold and spicy with a fun geometric print o...,Outdoor / Outdoor Décor / Outdoor Pillows & Cu...,2.0
4662,abbottsmoor dining chair cushion,the dining chair cushion ( set of 4 ) is apt f...,NaN,2.0
4663,zipparoll indoor chair cushion,zips from round pillow to flat pillow . the zi...,Kitchen & Tabletop / Tableware & Drinkware / T...,2.0
4664,indoor chair cushion,brighten your indoor seating area with this se...,NaN,2.0
4665,chair pad cushion,are your dining chair ’ s feeling a little sti...,NaN,2.0
4666,chair indoor seat cushion,add a splash of vibrant color and radiant styl...,NaN,2.0
4667,chair outdoor seat cushion,add a splash of vibrant color and radiant styl...,NaN,2.0
4668,dining chair cushion,add a splash of personality and create a cozy ...,Kitchen & Tabletop / Tableware & Drinkware / T...,2.0
4669,tropical outdoor lounge chair cushion,enhance your outdoor space with the addition o...,NaN,2.0


In [ ]:
graded_categorized[graded_categorized['query'] == QUERY][['product_name', 'product_description', 'category hierarchy', 'grade']]

,product_name,product_description,category hierarchy,grade
4690,replacement pillows outdoor lounge chair cushion,this replacement pillows outdoor lounge chair ...,NaN,2.0
4691,indoor/outdoor dining chair cushion and pillow...,go bold and spicy with a fun geometric print o...,Outdoor / Outdoor Décor / Outdoor Pillows & Cu...,2.0
4692,julep cushioned stackable chair,this stylish & playful chair was created by aw...,School Furniture and Supplies / School Furnitu...,0.0
4693,hard tufted vinyl chiavari chair cushion,hard cushions are the most popular choice in t...,Furniture / Office Furniture / Office Chair Ac...,2.0
4694,abbottsmoor dining chair cushion,the dining chair cushion ( set of 4 ) is apt f...,NaN,2.0
4695,zipparoll indoor chair cushion,zips from round pillow to flat pillow . the zi...,Kitchen & Tabletop / Tableware & Drinkware / T...,2.0
4696,indoor chair cushion,brighten your indoor seating area with this se...,NaN,2.0
4697,rotan 32 '' wide down cushion wingback chair,perfect for rounding out a living room seating...,Furniture / Living Room Furniture / Chairs & S...,0.0
4698,creative office chair pillow plush back seat c...,"the product is full filling , not afraid of ex...",Furniture / Office Furniture / Office Chair Ac...,2.0
4699,clarissa twin 32 '' wide pillow back futon chair,this convertible chair is ideal for small livi...,Furniture / Living Room Furniture / Chairs & S...,0.0
